# 🚀 Train YOLO26 & Export for Hailo-8L
This notebook demonstrates how to train a YOLO26 model on a custom dataset (e.g., Bottle Detection) and properly export the trained weights into an ONNX format compatible with the **Hailo Dataflow Compiler (DFC)**.

**Pipeline Overview:**
1. Install Dependencies
2. Configure Custom Dataset (`data.yaml`)
3. Train YOLO26 Nano
4. Evaluate Model Performance
5. Export to ONNX (with Hailo-specific opsets)
6. Verify Detections

In [ ]:
# Install the required Ultralytics package and visualization tools
!pip install ultralytics opencv-python matplotlib

## 1. Dataset Configuration
YOLO requires a `data.yaml` file to locate your training, validation, and testing images, as well as to map the class IDs to human-readable labels. 
Ensure your dataset folder structure looks like this:
```text
dataset/
├── train/
│   ├── images/
│   ├── labels/
│
|── valid/
│   ├── images/
│   ├── labels/
│
|── test/
|   ├── images/
|   ├── labels/
|
└── data.yaml



In [ ]:
import yaml

# Generate the data.yaml dynamically for the user
dataset_yaml = """
path: ./dataset        # Root directory of your dataset
train: train/images    # Train images (relative to 'path')
val: valid/images        # Validation images (relative to 'path')
test: test/images      # Test images (optional)

# Classes Configuration
names:
  0: bottle
# Add more classes here if needed (e.g., 1: cup, 2: person)
"""

with open("data.yaml", "w") as f:
    f.write(dataset_yaml)
    
print("✓ data.yaml successfully generated.")

## 2. Model Training
We will load the pre-trained `yolo26n.pt` weights and fine-tune them on our dataset. 
*Note: If you run out of GPU memory, reduce the `batch` size to 8.*

In [ ]:
from ultralytics import YOLO

# Load the pretrained YOLO26 Nano architecture
model = YOLO("yolo26n.pt")

# Train the model on the custom dataset
results = model.train(
    data="data.yaml",  
    epochs=100,        # Number of training epochs
    imgsz=640,         # Standard input dimension (640x640)
    batch=16,          # Batch size
    device=0,          # GPU index (0 for single GPU / Colab T4)
    plots=True         # Generate evaluation plots
)

print(f"✓ Training complete. Weights saved to: {results.save_dir}/weights/best.pt")

## 3. Evaluation Metrics
Let's visually inspect how well the model learned by plotting the generated Confusion Matrix from the validation phase.

In [ ]:
from IPython.display import Image, display
import os

# Display the confusion matrix to evaluate class accuracy
cm_path = os.path.join(results.save_dir, 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(filename=cm_path, width=800))
else:
    print("Confusion matrix plot not found.")

## 4. Export to ONNX (Critical for Hailo Compilation)
To compile this model into a `.hef` file using the Hailo toolchain, the ONNX export **must** meet specific structural requirements:
* **`opset=12`**: The Hailo parser relies on highly stable ONNX operators. Opset 12 is the safest standard.
* **`simplify=True`**: This removes redundant layers and fuses batch normalization, which is strictly required for edge acceleration mapping.
* **`imgsz=640`**: Hailo requires a static input tensor shape.

In [ ]:
# Load the best weights from our training run
best_model = YOLO(os.path.join(results.save_dir, "weights/best.pt"))

# Export the model to ONNX format
onnx_path = best_model.export(
    format="onnx",
    imgsz=640,       
    opset=12,        
    simplify=True    
)

print(f"✅ ONNX Export Complete: {onnx_path}")
print("You can now download this .onnx file and move to the 'export/' step of the repository!")

## 5. Verify Inference
Before downloading, let's run a quick sanity check on a few test images to ensure the ONNX model is accurately predicting bounding boxes.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Grab a few test images
test_images = glob.glob("./dataset/images/test/*.jpg")

if not test_images:
    print("No test images found. Make sure your dataset has a test/ folder.")
else:
    fig, axes = plt.subplots(1, len(test_images), figsize=(15, 5))
    
    # Handle single image case
    if len(test_images) == 1:
        axes = [axes]
        
    for i, img_path in enumerate(test_images):
        # Run inference using the trained weights
        res = best_model.predict(img_path, imgsz=640, conf=0.25)
        
        # Plot predictions on the image
        annotated_bgr = res[0].plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(annotated_rgb)
        axes[i].axis('off')
        axes[i].set_title(os.path.basename(img_path))
        
    plt.tight_layout()
    plt.show()